# Extract from Wikipedia

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("../input/wiki")
WIKI_PAGES = {
    "characters": "https://it.wikipedia.org/wiki/Personaggi_di_PK",
    "issues": "https://it.wikipedia.org/wiki/Albi_di_PK_-_Paperinik_New_Adventures",
    "quotes": "https://it.wikiquote.org/wiki/PK_-_Paperinik_New_Adventures",
}

In [16]:
import re
import requests


def make_download_url(page_url: str) -> str:
    """Return the MediaWiki API URL for the given page URL using regex extraction."""
    pattern = re.compile(r"^https://(?P<country>[a-z]{2})\.(?P<host>[a-z]+\.org)/wiki/(?P<title>[^?#]+)$")
    m = pattern.match(page_url)
    if not m:
        raise ValueError(f"Unsupported page URL format: {page_url}")
    country = m.group('country')
    host = m.group('host')
    title = m.group('title')  # already in underscore form
    api_url = (
        f"https://{country}.{host}/w/api.php?action=query&prop=revisions&titles={title}"
        f"&rvslots=main&rvprop=content&formatversion=2&format=json"
    )
    return api_url


def download_mediawiki(page_url: str) -> str:
    api_url = make_download_url(page_url)
    response = requests.get(api_url)
    response.raise_for_status()
    data = response.json()
    contents = data['query']['pages'][0]['revisions'][0]['slots']['main']['content']
    return contents

In [17]:
mediawiki_contents = {
    title: download_mediawiki(url)
    for title, url in WIKI_PAGES.items()
}

In [ ]:
import shutil
import subprocess


def mediawiki_to_md(mediawiki_contents: str) -> str:
    pandoc = shutil.which("pandoc")
    if not pandoc:
        raise RuntimeError("pandoc is not installed")
    proc = subprocess.run(
        [
            pandoc,
            "--from=mediawiki",
            "--to=markdown_strict",
            "--wrap=preserve",
            "--strip-comments",
            "--lua-filter=pandoc-strip-fmt.lua",
        ],
        input=mediawiki_contents,
        text=True,
        capture_output=True,
        check=True,
    )
    return proc.stdout


for title, content in mediawiki_contents.items():
    md = mediawiki_to_md(content)
    output_file = OUTPUT_DIR / f"{title}.md"
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(md)

In [28]:
import shutil
import subprocess


def mediawiki_to_txt(mediawiki_contents: str) -> str:
    pandoc = shutil.which("pandoc")
    if not pandoc:
        raise RuntimeError("pandoc is not installed")
    proc = subprocess.run(
        [
            pandoc,
            "--from=mediawiki",
            "--to=plain",
            "--wrap=preserve",
            "--lua-filter=pandoc-split-h.lua",
        ],
        input=mediawiki_contents,
        text=True,
        capture_output=True,
        check=True,
    )
    return proc.stdout.replace("{{'}}", "'")


for title, content in mediawiki_contents.items():
    txt = mediawiki_to_txt(content)
    output_file = OUTPUT_DIR / f"{title}.txt"
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(txt)